# Eight-Asset Portfolio Weighting and SPY Comparison

This Notebook constructs a ten-year investment portfolio from the eight selected holdings:

`SOXX, MRK, WMT, CVX, DUK, CME, LMT, PGR`

The user enters one non-negative coefficient for each holding. The coefficients are automatically normalized to sum to 100%. The default backtest applies **quarterly rebalancing at each quarter's first available closing date**, assumes dividends are reinvested through adjusted-close returns, and excludes transaction costs and taxes.

Each run creates a new timestamped subfolder under `portfolio results`, so earlier
runs are retained rather than overwritten.

Outputs:

1. a daily ten-year portfolio CSV;
2. a portfolio-versus-SPY summary, including monthly return correlation; and
3. an event table covering every SPY drawdown episode whose peak-to-trough decline exceeded 15%; and
4. a separate table for rapid SPY declines of at least 15% over 21 trading days (approximately one month).


## 1. Inputs and portfolio conventions


In [ ]:
# If required, remove the leading # and run this once:
# %pip install pandas numpy

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# The Notebook should be stored in the equity portfolio root folder.
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "portfolio results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = ["SOXX", "MRK", "WMT", "CVX", "DUK", "CME", "LMT", "PGR"]
BENCHMARK = "SPY"

# INPUT: enter any non-negative relative coefficients.
# Examples: eight values summing to 1, eight percentages summing to 100,
# or relative coefficients such as 2, 1, 1, 1, 1, 1, 1, 1.
WEIGHT_COEFFICIENTS = {
    "SOXX": 17,
    "MRK": 12,
    "WMT": 17,
    "CVX": 7,
    "DUK": 8,
    "CME": 10,
    "LMT": 9,
    "PGR": 20,
}

# Supported values: "none", "monthly", "quarterly", or "annual".
REBALANCE_FREQUENCY = "quarterly"
INITIAL_PORTFOLIO_VALUE = 100.0
SPY_DRAWDOWN_THRESHOLD = -0.15
SPY_ROLLING_WINDOW_DAYS = 21
SPY_ROLLING_RETURN_THRESHOLD = -0.15

print("Portfolio root:", BASE_DIR.resolve())
print("Output history folder:", OUTPUT_DIR.resolve())


## 2. CSV source mapping


In [ ]:
# Explicit source mapping keeps the audit trail clear and avoids choosing
# among the duplicate SPY and SOXX files downloaded for different comparisons.
CSV_PATHS = {
    "SOXX": BASE_DIR / "pharmaceutical" / "SOXX_10Y.csv",
    "MRK": BASE_DIR / "pharmaceutical" / "MRK_10Y.csv",
    "WMT": BASE_DIR / "consumer staples retail" / "WMT_10Y.csv",
    "CVX": BASE_DIR / "energy" / "CVX_10Y.csv",
    "DUK": BASE_DIR / "utilities" / "DUK_10Y.csv",
    "CME": BASE_DIR / "financial exchanges" / "CME_10Y.csv",
    "LMT": BASE_DIR / "aerospace defense" / "LMT_10Y.csv",
    "PGR": BASE_DIR / "property casualty insurance" / "PGR_10Y.csv",
    "SPY": BASE_DIR / "pharmaceutical" / "SPY_10Y.csv",
}

missing_files = [str(path) for path in CSV_PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(missing_files))

display(pd.DataFrame({"Ticker": CSV_PATHS.keys(), "CSV path": map(str, CSV_PATHS.values())}))


## 3. Validation, portfolio, and drawdown functions


In [ ]:
def load_adjusted_close(path, ticker):
    data = pd.read_csv(path, encoding="utf-8-sig")
    normalized = {
        " ".join(str(column).replace("_", " ").strip().lower().split()): column
        for column in data.columns
    }
    required = {"date": "Date", "adj close": "Adj Close"}
    missing = [label for key, label in required.items() if key not in normalized]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; found {list(data.columns)}")

    data = data.rename(columns={normalized[key]: label for key, label in required.items()})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    data["Adj Close"] = pd.to_numeric(data["Adj Close"], errors="coerce")

    if data["Date"].isna().any():
        raise ValueError(f"{path.name}: invalid dates detected.")
    if data["Date"].duplicated().any():
        raise ValueError(f"{path.name}: duplicate dates detected.")
    if data["Adj Close"].isna().any() or (data["Adj Close"] <= 0).any():
        raise ValueError(f"{path.name}: missing or non-positive adjusted prices detected.")

    series = data.sort_values("Date").set_index("Date")["Adj Close"]
    series.name = ticker
    return series


def normalize_coefficients(coefficients, asset_order):
    supplied = pd.Series(coefficients, dtype=float).reindex(asset_order)
    if supplied.isna().any():
        raise ValueError(f"A coefficient is required for every asset: {asset_order}")
    if (supplied < 0).any():
        raise ValueError("Long-only coefficients cannot be negative.")
    if supplied.sum() <= 0:
        raise ValueError("At least one coefficient must be positive.")
    return supplied / supplied.sum()


def rebalance_key(index, frequency):
    if frequency == "none":
        return pd.Series("all", index=index)
    if frequency == "monthly":
        return pd.Series(index.to_period("M").astype(str), index=index)
    if frequency == "quarterly":
        return pd.Series(index.to_period("Q").astype(str), index=index)
    if frequency == "annual":
        return pd.Series(index.year.astype(str), index=index)
    raise ValueError("REBALANCE_FREQUENCY must be none, monthly, quarterly, or annual.")


def simulate_target_weight_portfolio(prices, target_weights, initial_value, frequency):
    prices = prices.dropna(how="any").copy()
    if len(prices) < 2:
        raise ValueError("At least two aligned price observations are required.")

    shares = initial_value * target_weights / prices.iloc[0]
    nav_values = []
    realized_weights = []
    period_keys = rebalance_key(prices.index, frequency)

    for position, (date, row) in enumerate(prices.iterrows()):
        value_by_asset = shares * row
        portfolio_value = float(value_by_asset.sum())
        nav_values.append(portfolio_value)
        realized_weights.append((value_by_asset / portfolio_value).to_numpy())

        # Rebalance at that day's close when a new rebalance period begins.
        if position > 0 and period_keys.iloc[position] != period_keys.iloc[position - 1]:
            shares = portfolio_value * target_weights / row

    nav = pd.Series(nav_values, index=prices.index, name="Portfolio NAV")
    weight_history = pd.DataFrame(realized_weights, index=prices.index, columns=prices.columns)
    return nav, weight_history


def month_end_returns(nav):
    month_end = nav.groupby(nav.index.to_period("M")).last()
    return month_end.pct_change(fill_method=None).dropna()


def maximum_drawdown(nav):
    return float((nav / nav.cummax() - 1.0).min())


def cagr(nav):
    elapsed_years = (nav.index[-1] - nav.index[0]).days / 365.2425
    return float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / elapsed_years) - 1.0)


def find_spy_drawdown_episodes(spy_nav, portfolio_nav, threshold=-0.15):
    spy_nav = spy_nav.dropna()
    portfolio_nav = portfolio_nav.reindex(spy_nav.index).dropna()
    spy_nav = spy_nav.reindex(portfolio_nav.index)

    episodes = []
    in_drawdown = False
    peak_position = 0
    peak_value = float(spy_nav.iloc[0])

    for position in range(1, len(spy_nav)):
        current_value = float(spy_nav.iloc[position])

        if not in_drawdown:
            if current_value < peak_value:
                in_drawdown = True
            else:
                peak_position = position
                peak_value = current_value
        elif current_value >= peak_value:
            end_position = position
            episode_spy = spy_nav.iloc[peak_position:end_position + 1]
            episode_portfolio = portfolio_nav.iloc[peak_position:end_position + 1]
            spy_drawdown = episode_spy / episode_spy.iloc[0] - 1.0

            if float(spy_drawdown.min()) <= threshold:
                trough_position_local = int(np.argmin(spy_drawdown.to_numpy()))
                trough_date = episode_spy.index[trough_position_local]
                portfolio_to_trough = (
                    episode_portfolio.loc[trough_date] / episode_portfolio.iloc[0] - 1.0
                )
                portfolio_episode_dd = (
                    episode_portfolio / episode_portfolio.cummax() - 1.0
                ).min()
                episodes.append({
                    "SPY peak date": episode_spy.index[0],
                    "SPY trough date": trough_date,
                    "SPY recovery/end date": episode_spy.index[-1],
                    "Status": "Recovered",
                    "SPY maximum drawdown": float(spy_drawdown.min()),
                    "Portfolio return: peak to SPY trough": float(portfolio_to_trough),
                    "Portfolio maximum drawdown during episode": float(portfolio_episode_dd),
                    "Portfolio return: peak to recovery/end": float(
                        episode_portfolio.iloc[-1] / episode_portfolio.iloc[0] - 1.0
                    ),
                    "SPY return: peak to recovery/end": float(
                        episode_spy.iloc[-1] / episode_spy.iloc[0] - 1.0
                    ),
                    "Calendar days": int((episode_spy.index[-1] - episode_spy.index[0]).days),
                })

            peak_position = position
            peak_value = current_value
            in_drawdown = False

    # Include an unrecovered drawdown at the end of the sample.
    if in_drawdown:
        end_position = len(spy_nav) - 1
        episode_spy = spy_nav.iloc[peak_position:end_position + 1]
        episode_portfolio = portfolio_nav.iloc[peak_position:end_position + 1]
        spy_drawdown = episode_spy / episode_spy.iloc[0] - 1.0
        if float(spy_drawdown.min()) <= threshold:
            trough_position_local = int(np.argmin(spy_drawdown.to_numpy()))
            trough_date = episode_spy.index[trough_position_local]
            episodes.append({
                "SPY peak date": episode_spy.index[0],
                "SPY trough date": trough_date,
                "SPY recovery/end date": episode_spy.index[-1],
                "Status": "Ongoing at sample end",
                "SPY maximum drawdown": float(spy_drawdown.min()),
                "Portfolio return: peak to SPY trough": float(
                    episode_portfolio.loc[trough_date] / episode_portfolio.iloc[0] - 1.0
                ),
                "Portfolio maximum drawdown during episode": float(
                    (episode_portfolio / episode_portfolio.cummax() - 1.0).min()
                ),
                "Portfolio return: peak to recovery/end": float(
                    episode_portfolio.iloc[-1] / episode_portfolio.iloc[0] - 1.0
                ),
                "SPY return: peak to recovery/end": float(
                    episode_spy.iloc[-1] / episode_spy.iloc[0] - 1.0
                ),
                "Calendar days": int((episode_spy.index[-1] - episode_spy.index[0]).days),
            })

    return pd.DataFrame(episodes)


def find_rolling_stress_events(
    spy_nav,
    portfolio_nav,
    window_days=21,
    threshold=-0.15,
):
    # Group consecutive rolling-window breach dates into distinct stress events.
    pair = pd.concat(
        [spy_nav.rename("SPY"), portfolio_nav.rename("Portfolio")], axis=1
    ).dropna()
    spy_rolling_return = pair["SPY"] / pair["SPY"].shift(window_days) - 1.0
    portfolio_rolling_return = (
        pair["Portfolio"] / pair["Portfolio"].shift(window_days) - 1.0
    )
    breach = spy_rolling_return <= threshold
    breach_dates = pair.index[breach.fillna(False)]

    if len(breach_dates) == 0:
        return pd.DataFrame()

    # Consecutive trading-day breaches describe the same market shock.
    breach_positions = pair.index.get_indexer(breach_dates)
    groups = pd.Series(
        np.r_[True, np.diff(breach_positions) > 1], index=breach_dates
    ).cumsum()
    events = []
    for _, dates in pd.Series(breach_dates, index=breach_dates).groupby(groups):
        dates = pd.DatetimeIndex(dates.to_numpy())
        worst_date = spy_rolling_return.loc[dates].idxmin()
        worst_position = pair.index.get_loc(worst_date)
        start_position = worst_position - window_days
        start_date = pair.index[start_position]
        events.append({
            "Breach period start": dates.min(),
            "Breach period end": dates.max(),
            "Worst 21-day window start": start_date,
            "Worst 21-day window end": worst_date,
            "SPY return in worst window": float(spy_rolling_return.loc[worst_date]),
            "Portfolio return in same window": float(
                portfolio_rolling_return.loc[worst_date]
            ),
            "Portfolio minus SPY": float(
                portfolio_rolling_return.loc[worst_date]
                - spy_rolling_return.loc[worst_date]
            ),
            "Number of breach dates": int(len(dates)),
        })

    return pd.DataFrame(events)


## 4. Normalize weights and align all price series


In [ ]:
target_weights = normalize_coefficients(WEIGHT_COEFFICIENTS, ASSETS)
weight_table = pd.DataFrame({
    "Input coefficient": pd.Series(WEIGHT_COEFFICIENTS).reindex(ASSETS),
    "Normalized target weight": target_weights,
})
display(weight_table.style.format({"Normalized target weight": "{:.2%}"}))

all_prices = pd.concat(
    {
        ticker: load_adjusted_close(CSV_PATHS[ticker], ticker)
        for ticker in ASSETS + [BENCHMARK]
    },
    axis=1,
).sort_index()

rows_before_alignment = len(all_prices)
aligned_prices = all_prices.dropna(how="any")
if len(aligned_prices) < 200:
    raise ValueError("Too few common observations after aligning the nine price series.")

quality_summary = pd.DataFrame({
    "First date": [aligned_prices.index.min()],
    "Last date": [aligned_prices.index.max()],
    "Union date rows": [rows_before_alignment],
    "Common aligned rows": [len(aligned_prices)],
    "Rows removed by alignment": [rows_before_alignment - len(aligned_prices)],
})
display(quality_summary)
display(aligned_prices.head())
display(aligned_prices.tail())


## 5. Construct the portfolio and compare it with SPY


In [ ]:
portfolio_nav, realized_weights = simulate_target_weight_portfolio(
    aligned_prices[ASSETS],
    target_weights,
    INITIAL_PORTFOLIO_VALUE,
    REBALANCE_FREQUENCY,
)

spy_nav = aligned_prices[BENCHMARK] / aligned_prices[BENCHMARK].iloc[0] * INITIAL_PORTFOLIO_VALUE
spy_nav.name = "SPY NAV"

portfolio_daily_return = portfolio_nav.pct_change(fill_method=None)
spy_daily_return = spy_nav.pct_change(fill_method=None)
portfolio_drawdown = portfolio_nav / portfolio_nav.cummax() - 1.0
spy_drawdown = spy_nav / spy_nav.cummax() - 1.0

portfolio_output = pd.concat(
    [
        portfolio_nav,
        portfolio_daily_return.rename("Portfolio Daily Return"),
        portfolio_drawdown.rename("Portfolio Drawdown"),
        spy_nav,
        spy_daily_return.rename("SPY Daily Return"),
        spy_drawdown.rename("SPY Drawdown"),
    ],
    axis=1,
)
portfolio_output["SPY Drawdown Exceeds 15%"] = portfolio_output["SPY Drawdown"] <= SPY_DRAWDOWN_THRESHOLD
portfolio_output["SPY 21-Day Return"] = spy_nav / spy_nav.shift(SPY_ROLLING_WINDOW_DAYS) - 1.0
portfolio_output["SPY 21-Day Return At or Below -15%"] = (
    portfolio_output["SPY 21-Day Return"] <= SPY_ROLLING_RETURN_THRESHOLD
)
portfolio_output.index.name = "Date"

portfolio_monthly = month_end_returns(portfolio_nav)
spy_monthly = month_end_returns(spy_nav)
monthly_pair = pd.concat(
    [portfolio_monthly.rename("Portfolio"), spy_monthly.rename("SPY")], axis=1
).dropna()

summary = pd.DataFrame(
    {
        "Portfolio": {
            "10Y CAGR": cagr(portfolio_nav),
            "Annualized monthly volatility": portfolio_monthly.std(ddof=1) * np.sqrt(12),
            "Maximum drawdown": maximum_drawdown(portfolio_nav),
            "Correlation with SPY (monthly returns)": monthly_pair["Portfolio"].corr(monthly_pair["SPY"]),
            "Correlation with SPY (daily returns)": portfolio_daily_return.corr(spy_daily_return),
        },
        "SPY": {
            "10Y CAGR": cagr(spy_nav),
            "Annualized monthly volatility": spy_monthly.std(ddof=1) * np.sqrt(12),
            "Maximum drawdown": maximum_drawdown(spy_nav),
            "Correlation with SPY (monthly returns)": 1.0,
            "Correlation with SPY (daily returns)": 1.0,
        },
    }
)

display(summary.style.format("{:.2%}"))


## 6. Review SPY drawdown and rapid-decline stress events


In [ ]:
drawdown_episodes = find_spy_drawdown_episodes(
    spy_nav,
    portfolio_nav,
    threshold=SPY_DRAWDOWN_THRESHOLD,
)

if drawdown_episodes.empty:
    print("No SPY drawdown episode exceeded 15% in the aligned sample.")
else:
    formatted_episodes = drawdown_episodes.copy()
    percentage_columns = [
        "SPY maximum drawdown",
        "Portfolio return: peak to SPY trough",
        "Portfolio maximum drawdown during episode",
        "Portfolio return: peak to recovery/end",
        "SPY return: peak to recovery/end",
    ]
    for column in percentage_columns:
        formatted_episodes[column] = formatted_episodes[column].map(lambda value: f"{value:.2%}")
    display(formatted_episodes)

rolling_stress_events = find_rolling_stress_events(
    spy_nav,
    portfolio_nav,
    window_days=SPY_ROLLING_WINDOW_DAYS,
    threshold=SPY_ROLLING_RETURN_THRESHOLD,
)

print("\nRapid-decline test: SPY return <= -15% over 21 trading days")
if rolling_stress_events.empty:
    print("No rapid SPY decline met the configured threshold in the aligned sample.")
else:
    formatted_rolling_events = rolling_stress_events.copy()
    for column in [
        "SPY return in worst window",
        "Portfolio return in same window",
        "Portfolio minus SPY",
    ]:
        formatted_rolling_events[column] = formatted_rolling_events[column].map(
            lambda value: f"{value:.2%}"
        )
    display(formatted_rolling_events)


## 7. Export the CSV files


In [ ]:
# Create the run folder here so re-running this export cell also preserves history.
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_DIR / f"run_{run_stamp}"
suffix = 2
while RUN_DIR.exists():
    RUN_DIR = OUTPUT_DIR / f"run_{run_stamp}_{suffix:02d}"
    suffix += 1
RUN_DIR.mkdir(parents=False, exist_ok=False)

portfolio_csv = RUN_DIR / "investment_portfolio_10Y.csv"
summary_csv = RUN_DIR / "portfolio_vs_SPY_summary.csv"
episodes_csv = RUN_DIR / "SPY_drawdown_over_15pct_episodes.csv"
rolling_events_csv = RUN_DIR / "SPY_21_trading_day_decline_over_15pct_events.csv"
weights_csv = RUN_DIR / "normalized_portfolio_weights.csv"
settings_csv = RUN_DIR / "run_settings.csv"

portfolio_output.to_csv(portfolio_csv)
summary.to_csv(summary_csv)
drawdown_episodes.to_csv(episodes_csv, index=False)
rolling_stress_events.to_csv(rolling_events_csv, index=False)
weight_table.to_csv(weights_csv)

run_settings = pd.Series({
    "Run timestamp": datetime.now().isoformat(timespec="seconds"),
    "Rebalance frequency": REBALANCE_FREQUENCY,
    "Initial portfolio value": INITIAL_PORTFOLIO_VALUE,
    "SPY drawdown threshold": SPY_DRAWDOWN_THRESHOLD,
    "SPY rolling window (trading days)": SPY_ROLLING_WINDOW_DAYS,
    "SPY rolling-return threshold": SPY_ROLLING_RETURN_THRESHOLD,
    "Aligned first date": aligned_prices.index.min().date().isoformat(),
    "Aligned last date": aligned_prices.index.max().date().isoformat(),
}, name="Value")
run_settings.to_csv(settings_csv, header=True)

print("Generated run folder:", RUN_DIR.resolve())
print("Generated files:")
for path in [
    portfolio_csv,
    summary_csv,
    episodes_csv,
    rolling_events_csv,
    weights_csv,
    settings_csv,
]:
    print(" -", path)


## Interpretation conventions

- **Adjusted Close:** incorporates applicable splits and cash distributions, making it suitable for total-return analysis.
- **Weight coefficients:** automatically normalized; they do not need to sum to 1 or 100.
- **Quarterly rebalancing:** the portfolio is reset to its target weights at the close of the first available trading day in each new calendar quarter.
- **Correlation:** the headline figure uses month-end total returns, consistent with the earlier stock-selection analysis. Daily correlation is also reported as an audit statistic.
- **SPY drawdown episode:** begins at a prior SPY peak and ends when SPY recovers that peak, or at the sample end if recovery has not occurred. Only episodes with a peak-to-trough SPY decline of at least 15% are included.
- **No time limit for drawdown:** this is intentional and follows the standard peak-to-trough drawdown definition; it captures both fast crashes and slow bear markets.
- **Rapid-decline event:** separately identifies SPY losses of at least 15% over 21 trading days (approximately one month). Consecutive breach dates are grouped as one shock, and the worst 21-day window is reported. The 15% threshold is a configurable portfolio stress rule, not a universal regulatory standard.
- **Portfolio performance during stress:** reports the portfolio return from the SPY peak to its trough, the portfolio's own maximum drawdown during the complete episode, and its return through SPY recovery or the sample end.
- **Run history:** every execution writes to a new timestamped subfolder under `portfolio results`; previous runs are never intentionally overwritten.
- Transaction costs, taxes, slippage, and bid-ask spreads are excluded.
